# AlphaGenome-SAE: v2 SAE training on a Colab A100

Trains the residual-stream SAEs (resid_pre_b0/b4/b8) of one fold with the v2 recipe
(`ag_sae.train`), reading the activations already on Drive. Outputs go straight to Drive,
so a disconnected session resumes where it stopped: rerun the cells, finished seeds are
skipped and unfinished ones continue from `latest.pt`.

Runtime: **A100** GPU, **High-RAM**. Per tap the split is ~23.4 GiB (train) + 2.3 GiB (val);
on a 40 GB A100 the train split sits on the GPU, on an 80 GB one both do.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2; df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/alphagenome_sae'

## Settings
`CODE_SOURCE`: `"github"` clones the `a100-recipe-v2` branch (it must be pushed first);
`"drive"` copies the repo from `DRIVE/code/AlphaGenome-SAE`.

In [ ]:
FOLD = 3                     # 0-3
TAPS = ['resid_pre_b8', 'resid_pre_b4', 'resid_pre_b0']
POOLS = {                    # output pool -> extra train flags
    'sae_v2':     [],
    'sae_v2_k64': ['--topk', '64'],
}
SEEDS = [0, 1, 2]
PRECISION = 'tf32'           # 'bf16' is ~1.5-2x faster; compare in the pilot first
PLACEMENT = 'auto'         # 'host' if a 40 GB card runs out of memory
PILOT = True                 # first run: b8 seed0 only, legacy vs v2, k=64
CODE_SOURCE = 'github'       # or 'drive'

In [ ]:
import os, subprocess
if not os.path.isdir('/content/AlphaGenome-SAE'):
    if CODE_SOURCE == 'github':
        !git clone -b a100-recipe-v2 --depth 1 https://github.com/cetinazizm07/AlphaGenome-SAE.git /content/AlphaGenome-SAE
    else:
        !cp -r "{DRIVE}/code/AlphaGenome-SAE" /content/AlphaGenome-SAE
%cd /content/AlphaGenome-SAE
# Colab already has torch/numpy/pandas/pyarrow/scipy; only the package path is needed.
os.environ['PYTHONPATH'] = '/content/AlphaGenome-SAE'
!git log --oneline -1 2>/dev/null || true
!python -c "import ag_sae.train, torch; print('ok', torch.__version__, torch.cuda.get_device_name(0))"

## Activations: copy one tap at a time to local disk
Reading the shards once from local disk is faster and more robust than through the Drive mount.

In [ ]:
ACTS = f'/content/acts/fold{FOLD}'

def fetch(tap):
    os.makedirs(ACTS, exist_ok=True)
    src = f'{DRIVE}/acts/fold{FOLD}'
    !cp -n "{src}/index.json" "{ACTS}/"
    !rsync -a --info=progress2 --include="*_train_{tap}.npy" --include="*_val_{tap}.npy" --exclude="*" "{src}/" "{ACTS}/"

def drop(tap):
    !rm -f {ACTS}/*_{tap}.npy

def train(tap, pool, extra, seeds, recipe='v2', precision=PRECISION):
    out = f'{DRIVE}/{pool}/fold{FOLD}/{tap}_seed{{seed}}'
    cmd = ['python', '-u', '-m', 'ag_sae.train', '--activations', ACTS, '--tap', tap,
           '--out', out, '--n-features', '8192', '--recipe', recipe,
           '--precision', precision, '--placement', PLACEMENT, '--seeds', *map(str, seeds), *extra]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

## Pilot (~30-40 min)
b8 seed0 with k=64: the first campaign's recipe (exact fp32) against v2. Compare validation FVU,
never-fired latents and minutes before starting the full run.

In [ ]:
import json
if PILOT:
    fetch('resid_pre_b8')
    train('resid_pre_b8', 'pilot_legacy_k64', ['--topk', '64'], [0], recipe='legacy', precision='fp32')
    train('resid_pre_b8', 'pilot_v2_k64', ['--topk', '64'], [0])
    for pool in ('pilot_legacy_k64', 'pilot_v2_k64'):
        d = f'{DRIVE}/{pool}/fold{FOLD}/resid_pre_b8_seed0'
        r, h = json.load(open(f'{d}/training.json')), json.load(open(f'{d}/history.json'))[-1]['validation']
        print(f"{pool:>18}: fvu={r['best_validation_fvu']:.4f} never_fired={h['features_never_fired']} "
              f"dead={r['dead_latents']} {r['seconds']/60:.1f} min")

## Full run: every tap, every pool, three seeds

In [ ]:
for tap in TAPS:
    fetch(tap)
    for pool, extra in POOLS.items():
        train(tap, pool, extra, SEEDS)
    drop(tap)

In [ ]:
# Summary of what is on Drive for this fold.
import glob
for path in sorted(glob.glob(f'{DRIVE}/sae_v2*/fold{FOLD}/*/training.json')):
    r = json.load(open(path))
    print(f"{path.split('/')[-3]:>11} {path.split('/')[-2]:>20}: fvu={r['best_validation_fvu']:.4f} "
          f"dead={r['dead_latents']} {r['seconds']/60:.1f} min")